In [34]:
import tensorflow as tf

import pandas as pd

import re

import numpy as np

from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing.text import Tokenizer

from tensorflow.keras.preprocessing.sequence import pad_sequences

In [3]:
data = pd.read_csv("IMDB Dataset.csv")

data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [7]:
print(data.columns)

Index(['review', 'sentiment'], dtype='str')


In [8]:
print(data["sentiment"].unique())

<ArrowStringArray>
['positive', 'negative']
Length: 2, dtype: str


In [9]:
data["sentiment"] = data["sentiment"].str.strip()

In [16]:
texts = data["review"].astype(str).to_numpy()

labels = data["sentiment"].map({

    "positive":1,

    "negative":0

}).values

In [17]:
print(type(texts))
print(type(labels))

<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


In [31]:
print(data["sentiment"].value_counts())

sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [32]:
def clean_text(text):

    text = text.lower()

    text = re.sub(r"<br />", " ", text)

    text = re.sub(r"[^a-zA-Z]", " ", text)

    return text

In [35]:
texts = [clean_text(t) for t in texts]

In [36]:
X_train, X_test, y_train, y_test = train_test_split(

    texts,

    labels,

    test_size=0.2,

    random_state=42

)

In [37]:
tokenizer = Tokenizer(

    num_words=10000,

    oov_token="<OOV>"

)

tokenizer.fit_on_texts(X_train)

In [38]:
X_train_seq = tokenizer.texts_to_sequences(

    X_train

)

In [39]:
X_test_seq = tokenizer.texts_to_sequences(

    X_test

)

In [40]:
max_length = 200

In [41]:
X_train_pad = pad_sequences(

    X_train_seq,

    maxlen=max_length,

    padding='post',

    truncating='post'

)

In [42]:
X_test_pad = pad_sequences(

    X_test_seq,

    maxlen=max_length,

    padding='post',

    truncating='post'

)

In [43]:
print(X_train_pad.shape)

(40000, 200)


In [44]:
model = tf.keras.Sequential([

    tf.keras.layers.Embedding(

        input_dim=10000,

        output_dim=64,

        input_length=max_length

    ),

    tf.keras.layers.LSTM(64),

    tf.keras.layers.Dense(

        64,

        activation='relu'

    ),

    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(

        1,

        activation='sigmoid'

    )

])

In [45]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [46]:
model.compile(

    optimizer='adam',

    loss='binary_crossentropy',

    metrics=['accuracy']

)

In [47]:
history = model.fit(

    X_train_pad,

    y_train,

    validation_data=(

        X_test_pad,

        y_test

    ),

    epochs=5,

    batch_size=32

)

Epoch 1/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 63s 47ms/step - accuracy: 0.5368 - loss: 0.6875 - val_accuracy: 0.5118 - val_loss: 0.6929
Epoch 2/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 57s 45ms/step - accuracy: 0.5240 - loss: 0.6893 - val_accuracy: 0.5176 - val_loss: 0.6915
Epoch 3/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 57s 46ms/step - accuracy: 0.5595 - loss: 0.6728 - val_accuracy: 0.5185 - val_loss: 0.6919
Epoch 4/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 63s 50ms/step - accuracy: 0.6905 - loss: 0.5691 - val_accuracy: 0.8215 - val_loss: 0.4062
Epoch 5/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 60s 48ms/step - accuracy: 0.8630 - loss: 0.3412 - val_accuracy: 0.8692 - val_loss: 0.3241


In [48]:
model.save("sentiment_ai.h5")

In [55]:
while True:

    user_text = input(

        "Enter a review (or type quit): "

    )


    if user_text.lower() == "quit":

        break

In [63]:
    seq = tokenizer.texts_to_sequences(

        [user_text]

    )

In [64]:
    pad = pad_sequences(

        seq,

        maxlen=max_length,

        padding='post',

        truncating='post'

    )

In [65]:
    prediction = model.predict(pad)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step


In [66]:
    score = prediction[0][0]

    print("\nConfidence:", score)


Confidence: 0.9134088


In [67]:
    if score > 0.5:

        print("Prediction: Positive 😊\n")

    else:

        print("Prediction: Negative 😡\n")

Prediction: Positive 😊

